## imports

In [ ]:
import random

import copy
import os
import gc
import networkx as nx
import itertools
import numpy as np
import pandas as pd
import torch
from torch.nn import functional as F
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
from utils import min_max_normalize, get_cell_expression_data, get_data_matrix, to_entrez_id, \
    get_TCGA_co_abundance
import random
import pickle
from collections import defaultdict

from models.SPIDER import SPIDER
from dataset.SPIDER_dataset import SPIDERDataset

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


## Load data

In [ ]:
home_dir = '' # TODO: change to the path of the project

general_network_path = os.path.join(home_dir, 'data/networks/biogrid.net')
hek_network_path = os.path.join(home_dir, 'data/networks/hek_net.csv')
hct_network_path = os.path.join(home_dir, 'data/networks/hct_net.csv')
u2os_network_path = os.path.join(home_dir, 'data/networks/u2os_net.csv')

expression_path = os.path.join(home_dir, 'data/features/gene_expression/CCLE_expression.csv')
connections_methods_path = os.path.join(home_dir, 'data/features/methods.csv')
co_locations_path = os.path.join(home_dir, 'data/features/localization/co_locations_tmp.csv')
proteins_expression_path = os.path.join(home_dir, 'data/features/protein_expression')

connections_methods = pd.read_csv(connections_methods_path)
connections_methods.columns.values[0] = 'g1'
connections_methods.columns.values[1] = 'g2'
cols = connections_methods.columns[2:]
connections_methods[cols] = connections_methods[cols].astype(float)
connections_methods.iloc[:, 2:] = min_max_normalize(connections_methods.iloc[:, 2:])
connections_methods_dict = \
    dict(zip(zip(connections_methods.g1, connections_methods.g2),
              [t[3:] for t in connections_methods.itertuples()]))

gene_expression = pd.read_csv('data/features/gene_expression.csv', index_col=0)
hek_expression_data = gene_expression[['HEKTE']]
hct_expression_data = gene_expression[['HCT116']]
u2os_expression_data = gene_expression[['U2OS']]

general_network = pd.read_csv(general_network_path, header=0)
general_network = general_network[['g1', 'g2']].drop_duplicates()
general_network = general_network[pd.to_numeric(general_network['g1'], errors='coerce').notnull()]
general_network = general_network[pd.to_numeric(general_network['g2'], errors='coerce').notnull()].astype(int)

location_data = pd.read_csv(os.path.join(home_dir, 'data/features/localization/go_locations.csv'))
location_data = location_data.groupby('Gene').agg('mean')

co_locations_data = pd.read_csv(co_locations_path).rename(columns={'0': 'g1', '1': 'g2'}).set_index(['g1', 'g2'])
hek_co_abundance_path = os.path.join(home_dir, proteins_expression_path, 'hek_co_abundance.csv')
hct_co_abundance_path = os.path.join(home_dir, proteins_expression_path, 'hct_co_abundance.csv')
hek_co_abundance = pd.read_csv(hek_co_abundance_path).rename(columns={'0': 'g1', '1': 'g2'}).set_index(['g1', 'g2'])
hct_co_abundance = pd.read_csv(hct_co_abundance_path).rename(columns={'0': 'g1', '1': 'g2'}).set_index(['g1', 'g2'])

proteins_expressions = {t.split('.')[0].split('_')[-1].lower(): pd.DataFrame(min_max_normalize(pd.read_csv(f'{proteins_expression_path}/{t}', header=0).groupby('0').agg('mean')))
            for t in ['hek.csv', 'hct.csv']}

hek_network = pd.read_csv(hek_network_path, header=0)
hct_network = pd.read_csv(hct_network_path, header=0)
u2so_data = pd.read_csv(os.path.join(home_dir, 'data/networks/U2OS_net.csv'))[['name']]
u2so_edges = [(e.split(' ')[0], e.split(' ')[-1]) for e in u2so_data.name]
u2so_network = pd.DataFrame(u2so_edges, columns=['g1', 'g2'])

In [ ]:
expression_len = 2
prots_len = 3
locations_len = 2 * location_data.shape[1] + 1
methods_len = 11
prot_included = True  # TODO: change to False if not using proteins

## Train model

In [ ]:
# create train, val, test datasets

from typing import Any


random.seed(1)
np.random.seed(1)

cell1, cell2 = ['hek', 'hct']  # TODO: change if needed
nodes = list(set([int(n) for n in location_data.index]))
test_size = int(0.25 * len(set(nodes)))
val_size = int(0.25 * len(set(nodes)))

test_nodes =  set(random.sample(nodes, test_size))
nodes = list(set(nodes) - test_nodes)
val_nodes = set(random.sample(nodes, val_size))
train_nodes = set(nodes) - val_nodes
train_interactions = general_network[(general_network['g1'].isin(train_nodes)) & (general_network['g2'].isin(train_nodes))]
val_interactions = general_network[(general_network['g1'].isin(val_nodes)) & (general_network['g2'].isin(val_nodes))]
test_interactions = general_network[(general_network['g1'].isin(test_nodes)) & (general_network['g2'].isin(test_nodes))]

print(len(train_interactions), len(val_interactions), len(test_interactions))
if prot_included:
    datasets = {
        'hek': (hek_network, hek_expression_data, proteins_expressions[cell1], hek_co_abundance),
        'hct': (hct_network, hct_expression_data, proteins_expressions[cell2], hct_co_abundance),
    }
else:
    datasets = {
        'hek': (hek_network, hek_expression_data, None, None),
        'hct': (hct_network, hct_expression_data, None, None),
        'u2os': (u2so_network, u2os_expression_data, None, None),
    }

cells_data = defaultdict(list)
cells = ['hek', 'hct', 'u2os'] if not prot_included else ['hek', 'hct']
for cell in cells:
    print(cell)
    for interactions, dataset_name in zip([train_interactions, val_interactions, test_interactions], ['train', 'val', 'test']):
        network1, expression_data1, protein_data1, co_abundance1 = datasets[cell]
        data, labels, edges = get_data_matrix(
            interactions, network1, gene_expression=expression_data1,
            prot_expression=protein_data1, co_abundance=co_abundance1,
            locations=location_data, methods=connections_methods_dict, co_locations=co_locations_data)
        data['edges'] = [str(e) for e in edges]
        cells_data[cell].append(pd.DataFrame(data))
        cells_data[cell].append(list(labels))
    cells_data[cell] = tuple(cells_data[cell])

with open('data/cells_data.pkl', 'wb') as f:
    pickle.dump(cells_data, f)



In [ ]:
# train SPIDER - hek293t


X_train, y_train, X_val, y_val, X_test, y_test = copy.deepcopy(cells_data['hek'])

params = [True] * 6 if prot_included else [True, False, False, True, True, True]
for l, d, y in zip(['train', 'val', 'test'], [X_train, X_val, X_test], [y_train, y_val, y_test]):
    datasets[l] = SPIDERDataset(data=d, y_data=y, expression_size=expression_len, 
                                prot_size=3, locations_size=locations_len, methods_size=methods_len, 
                                params=params)

torch.manual_seed(seed=1234)
np.random.seed(1234)
spider_model = SPIDER(expression_size=expression_len, locations_size=locations_len, 
                prot_size=prots_len, graph_matrix=datasets['train'].graph_matrix,
                second_input_size=datasets['train'].interaction.shape[-1],
                hidden_size=64, p=0.3, params=params)
spider_model.train_all(datasets, epochs=750)
spider_model.eval()

# SPIDER evaluation - first set the graph matrix to the test set
spider_model.graph_matrix = datasets['test'].graph_matrix
y_preds = spider_model((datasets['test'].interaction, datasets['test'].edge_index))
y_true = datasets['test'].y.cpu().detach().numpy()
y_scores = y_preds.cpu().detach().numpy()
precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
print('PR score:', auc(recall, precision))
print('ROC score:', roc_auc_score(y_true, y_scores))
fscore = (2 * precision * recall) / (precision + recall + 1e-6)
ix = np.argmax(fscore)
print('max fscore:', fscore[ix])

os.makedirs(os.path.join(home_dir, 'spider_models'), exist_ok=True)
if prot_included:
    torch.save(spider_model, os.path.join(home_dir, 'spider_models', 'hek.pt'))
else:
    torch.save(spider_model, os.path.join(home_dir, 'spider_models', 'hek_no_prots.pt'))


## Run model - TCGA Example

In [ ]:
# if prot_included:
#     spider_model = torch.load(os.path.join(home_dir, 'spider_models', 'hek.pt'), map_location=DEVICE)
# else:
#     spider_model = torch.load(os.path.join(home_dir, 'spider_models', 'hek_no_prots.pt'), map_location=DEVICE)

In [ ]:
import requests

url = "https://datahub.assets.cbioportal.org/brca_tcga_pan_can_atlas_2018.tar.gz"
output_path = "brca_tcga_pan_can_atlas_2018.tar.gz"

print(f"Downloading TCGA BRCA data from {url} ...")

with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open(output_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

print(f"Download complete. File saved as {output_path}.")

In [ ]:
! tar -xzf brca_tcga_pan_can_atlas_2018.tar.gz
! mkdir -p data/features/TCGA/BRCA
! mv brca_tcga_pan_can_atlas_2018/data_mrna_seq_v2_rsem.txt data/features/TCGA/BRCA
! mv brca_tcga_pan_can_atlas_2018/data_protein_quantification.txt data/features/TCGA/BRCA
! rm -rf brca_tcga_pan_can_atlas_2018
! rm brca_tcga_pan_can_atlas_2018.tar.gz

In [ ]:
BRCA_protein_expression_path = 'data/features/TCGA/BRCA/data_protein_quantification.txt'
BRCA_protein_expression_data = pd.read_csv(BRCA_protein_expression_path, sep='\t').rename(columns={'Composite.Element.REF': 'gene'}).set_index('gene').fillna(0)
symbol_to_entrez = to_entrez_id(list(BRCA_protein_expression_data.index.str.split('|').str[0].str.strip()), scopes='symbol')
BRCA_protein_expression_data.index = BRCA_protein_expression_data.index.str.split('|').str[0].str.strip().map(symbol_to_entrez)
BRCA_protein_expression_data = min_max_normalize(BRCA_protein_expression_data)

BRCA_gene_expression_path = 'data/features/TCGA/BRCA/data_mrna_seq_v2_rsem.txt'
BRCA_gene_expression_data = pd.read_csv(BRCA_gene_expression_path, sep='\t').rename(columns={'Entrez_Gene_Id': 'gene'}).set_index('gene')  #.fillna(0) # [protein_expression_data.columns]
BRCA_gene_expression_data = min_max_normalize(np.log10(BRCA_gene_expression_data.iloc[:, 1:] + 1))

mutual_samples = list(set(BRCA_gene_expression_data.columns) & set(BRCA_protein_expression_data.columns))

BRCA_gene_expression_data = BRCA_gene_expression_data[mutual_samples]
BRCA_protein_expression_data = BRCA_protein_expression_data[mutual_samples]

In [ ]:
from tqdm import tqdm

# calculated co-abundances are at: https://drive.google.com/file/d/1K47omv5UcMaB_a9BRUxyXCqoipvHMgNz/view?usp=sharing


all_networks = None
params = [True, True, True, True, True, True]
for sample in list(mutual_samples)[:10]:  # Only calculate the first 10 networks for the sake of the example
    print(f'Calculating network for {sample}')
    if os.path.exists(f'data/features/TCGA/BRCA/co_abundance_{sample}.csv'):
        co_abundance = pd.read_csv(f'data/features/TCGA/BRCA/co_abundance_{sample}.csv').set_index(['g1', 'g2'])
    else:
        co_abundance = get_TCGA_co_abundance(BRCA_gene_expression_data, sample, general_network).set_index(['g1', 'g2'])
        co_abundance.to_csv(f'data/features/TCGA/BRCA/co_abundance_{sample}.csv')
    data, _, edges = get_data_matrix(
        general_network, gene_expression=BRCA_gene_expression_data[[sample]], prot_expression=BRCA_protein_expression_data[[sample]],
        locations=location_data, co_abundance=co_abundance, methods=connections_methods_dict, co_locations=co_locations_data, 
        gen_labels=False)
    data.insert(2, 'p1', np.nan)
    data.insert(3, 'p2', np.nan)
    data.insert(4, 'cop', np.nan)
    edges = [str(e) for e in edges]
    data['edges'] = edges
    sample_dataset = SPIDERDataset(data=data, y_data=None, expression_size=expression_len, 
            locations_size=locations_len, 
            methods_size=11, params=params)
    
    spider_model.graph_matrix = copy.deepcopy(sample_dataset.graph_matrix)
    x = (sample_dataset.interaction.to(DEVICE), sample_dataset.edge_index.to(DEVICE))
    preds = spider_model(x)
    del x
    sample_df = pd.DataFrame(preds.cpu().detach().numpy(), columns=['weight'])
    sample_df['source'] = [int(e.split(', ')[0].split('(')[-1]) for e in data.iloc[:, -1]]
    sample_df['target'] = [int(e.split(', ')[-1].split(')')[0]) for e in data.iloc[:, -1]]
    if all_networks is None:
        all_networks = sample_df.rename(columns={'weight': sample})
    else:
        all_networks = pd.merge(all_networks, sample_df.rename(columns={'weight': sample}), on=['source', 'target'], how='outer')
    del preds
    del sample_dataset
    torch.cuda.empty_cache()
    gc.collect()
